# JUICE RPWI HF SID9 (PSSR2-S-single): L1a QL -- 2026/7/22

# Import lib

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Library setting

In [ ]:
# The latest CDF library https://spdf.gsfc.nasa.gov/pub/software/cdf/dist/latest/
from spacepy import pycdf

import os
os.environ["CDF_LIB"] = "/Applications/cdf/cdf39_1-dist/lib"

# *** Library ***
sys.path.append('./lib/')
import juice_sid22_lib as juice_sid22

# Mode seting

In [ ]:
# *** Mode set ***
mode_dump = 0                           # 0: no-dump  1:plot dump
# *** Dump ***
mode_unit = 0                           # [Power]     0: raw     1: V＠ADC   2: V@HF    3: V@RWI    >=4: V/m
mode_band = 0                           # [Power]     0: sum     1: /Hz
mode_bg   = 3                           # [Power]     0: BG      1: CAL      2: off   >=3: all

# *** Directory set: set by User ***
work_dir = '/Users/user/0-python/JUICE_python/ql/'                         # Plot dump folder

# get CDF data

In [ ]:
date='20260716';  ver = 'V01'   # ASW3 / PC4
# date='20260720';  ver = 'V01'   # ASW3 / PC4
date='0';         ver = ''      # Test data
sid = 9               # 6 / 9 / 22
data_dir, data_name_list = juice_sid22.datalist(date, ver, sid)       # [date]   yyyymmdd: group read    others: file list

In [ ]:
class struct:
    pass

data = struct()
num_list = len(data_name_list)
for i in range(num_list):
    data_name = data_name_list[i];  cdf_file = data_dir + data_name;  print(i, cdf_file)
    cdf = pycdf.CDF(cdf_file);      data1 = juice_sid22.hf_sid22_read(cdf)
    if i==0:
        data = data1;                                       print("sid", sid, data.auto_corr.shape)
    else:
        data = juice_sid22.hf_sid22_add(data, data1);  print("sid", sid, data.auto_corr.shape)
data_name = os.path.split(data_name)[1];                    print("data name:", data_name)

In [ ]:
data = juice_sid22.hf_sid22_shaping(data, sid)

date1 = data.epoch[0];  date1 = date1.strftime('%Y/%m/%d %R:%S')
date2 = data.epoch[-1]; date2 = date2.strftime('%Y/%m/%d %R:%S')
str_date = date1 + "  -  " + date2;  print(str_date)
n_time = data.auto_corr.shape[0]
n_time0 = n_time-1

In [ ]:
if data.n_time>1: 
    date3 = data.epoch[1];  date3 = date3.strftime('%Y/%m/%d %R:%S')
    print("       date and time:", str_date, "(interval:", data.epoch[1] - data.epoch[0], date3, ")")
else:
    print("       date and time:", str_date)
print("data size:", data.auto_corr.shape)
print("==> Ch:", data.ch_selected[0], "  Num-sweep:", n_time, "   Num-AutoCorr: 16    each Length:", data.N_lag[0] )

# Raw data

In [ ]:
fig = plt.figure(figsize=(16, 11))
ax1 = fig.add_subplot(6, 1, 1);  ax2 = fig.add_subplot(6, 1, 2);  ax3 = fig.add_subplot(6, 1, 3)
ax4 = fig.add_subplot(6, 1, 4);  ax5 = fig.add_subplot(6, 1, 5);  ax6 = fig.add_subplot(6, 1, 6)

ax1.plot(np.ravel(data.auto_corr[:][:]),     linewidth=.5, label='auto_corr');    ax1.set_ylabel('AutoCorr')
ax2.plot(np.ravel(data.time[:][:]),          linewidth=.5, label='time');         ax2.set_ylabel('time')
ax3.plot(np.ravel(data.EE[:][:]),     '-y',  linewidth=2,  label='EE');           ax3.set_ylabel('EE');           ax3.set_yscale('log')
ax3.plot(np.ravel(data.EE_raw[:][:]), '-r',  linewidth=1,  label='EE_raw')
ax3.plot(np.ravel(data.EE_amp[:][:]), '--b', linewidth=1,  label='EE_amp')
ax4.plot(np.ravel(data.frequency[:]),        linewidth=.5, label='frequency');    ax4.set_ylabel('Frequency [kHz]')
ax5.plot(np.ravel(data.cal_signal*10), '-k', label='CAL*10')
ax5.plot(np.ravel(data.freq_ID),       '-r', label='freq_ID')
ax5.plot(np.ravel(data.proc_param1),   '-g', label='param1:4 - wide&short[dec:0], 6-narrow&long[dec:1]')
ax6.plot(np.ravel(data.epoch[:]),      '.');                                      ax6.set_ylabel('Date-Time');    ax6.set_xlabel(str_date)
#
title_label = '[JUICE/RPWI HF PSSR2-S-single (SID-9)]\n' + data_name;  ax1.set_title(title_label)
ax3.legend(loc='upper right', fontsize=8);   ax5.legend(loc='upper right', fontsize=8);  
#
xlim=[-.5, len(np.ravel(data.auto_corr))-.5];  print(xlim);  ax1.set_xlim(xlim);  ax2.set_xlim(xlim)
xlim=[-.5, len(np.ravel(data.frequency))-.5];  print(xlim);  ax3.set_xlim(xlim);  ax4.set_xlim(xlim)
xlim=[-.5, len(np.ravel(data.epoch))    -.5];  print(xlim);  ax5.set_xlim(xlim);  ax6.set_xlim(xlim)

fig.show
if mode_dump == 1:
    png_fname = work_dir+data_name+'_raw.png'
    fig.savefig(png_fname)

## First data

In [ ]:
data.auto_corr.shape, data.freq_ID, data.frequency.shape

In [ ]:
n_sweep = 0
n_lag0 = 4

fig = plt.figure(figsize=(14, 15))
ax1  = fig.add_subplot(2, 1, 1);  ax2  = fig.add_subplot(2, 1, 2)
ax1.plot(data.time[n_sweep][n_lag0:], data.auto_corr[n_sweep][n_lag0:], linewidth=.5)
ax2.plot(data.frequency[n_sweep], data.EE[n_sweep], '.r', linewidth=.5)

date1 = data.epoch[n_sweep];  date1 = date1.strftime('First: %Y/%m/%d %R:%S')
title_date = "[" + data_name + "]\n" + date1;  ax1.set_title(title_date)
#
ax1.set_xlabel("{0:.0f} [kHz]".format(data.frequency[n_sweep][data.freq_ID[0]]))
ax2.set_xlabel("frequency [kHz]");  

fig.subplots_adjust(hspace=.1)
fig.show
if mode_dump == 1:
    png_fname = work_dir+data_name+'_1st.png'
    fig.savefig(png_fname)

## Mid Data

In [ ]:
n_sweep = np.int16(n_time0/2)
n_lag0 = 4

fig = plt.figure(figsize=(14, 15))
ax1  = fig.add_subplot(2, 1, 1);  ax2  = fig.add_subplot(2, 1, 2)
ax1.plot(data.time[n_sweep][n_lag0:], data.auto_corr[n_sweep][n_lag0:], linewidth=.5)
ax2.plot(data.frequency[n_sweep], data.EE[n_sweep], '.r', linewidth=.5)

date1 = data.epoch[n_sweep];  date1 = date1.strftime('First: %Y/%m/%d %R:%S')
title_date = "[" + data_name + "]\n" + date1;  ax1.set_title(title_date)
#
ax1.set_xlabel("{0:.0f} [kHz]".format(data.frequency[n_sweep][data.freq_ID[0]]))
ax2.set_xlabel("frequency [kHz]");  

fig.subplots_adjust(hspace=.1)
fig.show
if mode_dump == 1:
    png_fname = work_dir+data_name+'_1st.png'
    fig.savefig(png_fname)

## Last data

In [ ]:
n_sweep = n_time0
n_lag0 = 4

fig = plt.figure(figsize=(14, 15))
ax1  = fig.add_subplot(2, 1, 1);  ax2  = fig.add_subplot(2, 1, 2)
ax1.plot(data.time[n_sweep][n_lag0:], data.auto_corr[n_sweep][n_lag0:], linewidth=.5)
ax2.plot(data.frequency[n_sweep], data.EE[n_sweep], '.r', linewidth=.5)

date1 = data.epoch[n_sweep];  date1 = date1.strftime('First: %Y/%m/%d %R:%S')
title_date = "[" + data_name + "]\n" + date1;  ax1.set_title(title_date)
#
ax1.set_xlabel("{0:.0f} [kHz]".format(data.frequency[n_sweep][data.freq_ID[0]]))
ax2.set_xlabel("frequency [kHz]");  

fig.subplots_adjust(hspace=.1)
fig.show
if mode_dump == 1:
    png_fname = work_dir+data_name+'_1st.png'
    fig.savefig(png_fname)

## Overall

In [ ]:
p_max = np.max(data.EE);  p_min = np.min(data.EE)
EE_2d = data.EE.transpose()
num_x = np.arange(data.n_time);     num_y = data.frequency[0]

fig2d = plt.figure(figsize=[16,11])
ax1 = fig2d.add_subplot(1, 1, 1);   ax1.set_yscale('log'); ax1.set_title("[" + data_name + "]")
p1  = ax1.pcolormesh(num_x, num_y, EE_2d, norm=colors.LogNorm(vmin=p_min, vmax=p_max), cmap='jet');  pp1 = fig2d.colorbar(p1, ax=ax1, orientation="vertical")

fig2d.show
if mode_dump == 1:
    png_fname = work_dir+data_name+'_overall_E.png'
    fig2d.savefig(png_fname)

In [ ]:
fig = plt.figure(figsize=(14, 8));  ax1  = fig.add_subplot(1, 1, 1)
ax1.plot(num_x, np.median(data.EE_raw, axis=1), '-k',  linewidth=2,        label='EE_raw median'); 
ax1.plot(num_x, np.median(data.EE_amp, axis=1), '.k',  linewidth=1, ms=10, label='EE_amp median')
ax1.plot(num_x, np.median(data.EE,     axis=1), '--k', linewidth=2,        label='EE     median')
ax1.legend(loc='upper right', fontsize=8);  ax1.set_yscale('log')
date1 = data.epoch[n_sweep];  date1 = date1.strftime('First: %Y/%m/%d %R:%S');  title_date = "[" + data_name + "]\n" + date1;  ax1.set_title(title_date)
ax2.set_xlabel("frequency [kHz]");  
ylim=[10**-4, 10**4];  ax1.set_ylim(ylim)
fig.subplots_adjust(hspace=.1)
fig.show

In [ ]:
data.auto_corr.shape, n_time, data.N_step[0], data.N_lag[0], data.N_step[0] * data.N_lag[0]

In [ ]:
# print(data.auto_corr.shape, data.n_time, data.N_step[0], data.N_lag[0])
auto_corr_2d = data.auto_corr.reshape(n_time, data.N_lag[0])
auto_corr_2d = auto_corr_2d.transpose()
print(data.auto_corr.shape, data.n_time, data.N_step[0], data.N_lag[0], auto_corr_2d.shape)

num_x = np.arange(data.n_time)
num_y = np.arange(data.N_lag[0])

fig2d = plt.figure(figsize=[16,11])
ax1 = fig2d.add_subplot(1, 1, 1); 
ax1.set_title("[" + data_name + "]")
p1 = ax1.pcolormesh(num_x, num_y, auto_corr_2d, cmap='bwr')
pp1 = fig2d.colorbar(p1, ax=ax1, orientation="vertical")

# Plot
fig2d.show
if mode_dump == 1:
    png_fname = work_dir+data_name+'_overall.png'
    fig2d.savefig(png_fname)

In [ ]:
##### HF-QF                   [b0-1:0x03 -- Error Data]
#####   b0: RWI-OFF            0:on    1:OFF 
#####   b1: overflow in data   0:n/a   1:Overflow
#####   b2-3:overflow in ADC   0:n/a   1:few OVF   2: OVF>1%   3: OVF>10%
#####   b4: cal_signal         0:non   1:CAL
#####   b5: RIME               0:n/a   1:Activated
#####   b6: SuvHeater          0:none  1:ON    
#####   b7: Eclipse            0:n/a   1:Eclipse
# QF    array([ 5,  5,  5,  5,  5,  5,  5,  5, 21, 21, 21, 21,  5,  5,  5,  5,  5, 5,  5,  5,  5,  5,  5,  5, 21, 21, 21, 21,  5,  5,  5,  5,  5],
# OVF   array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=uint8),
# CAL   array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0], dtype=uint8))
data.gain_raw, data.df_raw, data.HF_QF, data.ADC_ovrflw, data.cal_signal

In [ ]:
data.epoch